# Chapter 1 — memorisation vs relational knowledge

**Settings:** GPU **T4 ×2** · Internet **ON** · Persistence **Variables and Files**

> Nothing to attach. Section 2 downloads YAGO3-10 from KG-LLM's repo.

---

### The claim

> Fine-tuning for KGC installs **entity-name memorisation**, not relational structure.
> Measured so far: memorisation **0.393** of the 0.4315 above-chance gain — **91%**.

### Three rules learned the hard way

**1 · One GPU per job.** Two visible → `DataParallel` → autocast never reaches the replicas → `mat1 and mat2 must have the same dtype`. Every command below is pinned.

**2 · fp16 + `sdpa`.** fp16 + `eager` returns **NaN** on Qwen2.5 — it looks like `train_loss=0.0` with `grad_norm=nan`, i.e. a finished run.

**3 · Remove `torchao`.** Kaggle ships 0.10.0; transformers refuses to import below 0.16. Nothing here uses it.

### This chapter needs only LoRA — no MoRA fork, no BOFT kernel

So it runs in **one** session with official peft and none of the environment conflicts that blocked Chapters 2 and 3.

## 0 · Setup

In [1]:
REPO_URL = "https://github.com/lynda-lagh/contribution-.git"
DEST = "/kaggle/working/repo"

import os, subprocess, sys, socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No network. Settings > Internet > ON, then re-run.")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f"{DEST}/.git"):
    run(["git", "-C", DEST, "fetch", "--all"])
    run(["git", "-C", DEST, "reset", "--hard", "origin/main"])
    print("updated")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, DEST])
    print("cloned")

os.chdir(DEST); sys.path.insert(0, DEST)
print("HEAD:", run(["git", "log", "-1", "--oneline"]))
print("\n\u2605 Does that hash match your latest push?")

cloned
HEAD: ded69bc chapter1 update

★ Does that hash match your latest push?


In [2]:
PIN_TRANSFORMERS = "4.57.6"

import json, subprocess, sys
def stack():
    code = ("import json, peft, transformers; print(json.dumps({"
            "'peft': peft.__version__, 'tf': transformers.__version__}))")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None

s = stack()
if not (s and s["tf"] == PIN_TRANSFORMERS):
    print(f"installing… (have {s})")
    !pip install -q -r requirements.txt
    !pip install -q peft "transformers=={PIN_TRANSFORMERS}"

# torchao 0.10.0 blocks the transformers import and kills LoRA. Nothing uses it.
!pip uninstall -y -q torchao 2>/dev/null

import peft, transformers
print(f"\npeft {peft.__version__} | transformers {transformers.__version__}")

installing… (have {'peft': '0.19.1', 'tf': '5.0.0'})
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 625.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 38.8 MB/s

### 0b · Save / restore — **read this before you run anything**

Kaggle deletes `/kaggle/working` when the session ends unless you press **Save Version**. `save_all()` writes one zip at the top level, outside the git clone, holding everything expensive: adapters, the YAGO type file, all results, and a record of which steps finished.

It is called automatically after every real step below.

In [ ]:
# ═══ 0b · SAVE / RESTORE — run this before anything else ═════════════════
# WHY THIS EXISTS. Two separate things eat your work:
#   1. checkpoints/ is gitignored AND sits inside the git clone at
#      /kaggle/working/repo, so `git reset --hard` in cell 2 wipes it.
#   2. Kaggle throws away /kaggle/working unless you press "Save Version".
#      Turning the session off discards it silently — that is the 84 KiB you
#      saw last time.
# Writing to the TOP level of /kaggle/working fixes (1). Only YOU fix (2).
import json, zipfile, glob, shutil, time, datetime
from pathlib import Path

REPO = Path("/kaggle/working/repo")
OUT  = Path("/kaggle/working")
STATE = OUT / "ch1_state.json"

# what to keep, ordered by how painful it is to lose ----------------------
KEEP = [
    # (label, glob patterns relative to REPO, why it hurts to lose)
    ("adapters", ["checkpoints/*/adapter_model.safetensors",
                  "checkpoints/*/adapter_model.bin",
                  "checkpoints/*/adapter_config.json",
                  "checkpoints/*/train_summary.json",
                  "checkpoints/*/tokenizer*", "checkpoints/*/special_tokens*"],
     "GPU-HOURS. Cannot be recreated without retraining."),
    ("types",    ["data/*/entity2type.txt"],
     "a multi-GB download + a full pass over the YAGO dump."),
    ("results",  ["results/*.json", "results/*.md", "results/*.tex"],
     "every number, every figure input, the plain-English report."),
    ("manifests", ["data/*/built/manifest.json"],
     "provenance: seed, condition, type source, positive rate."),
]

def _mark(step: str) -> None:
    """Remember which steps finished, so a restored session knows where it was."""
    s = json.loads(STATE.read_text()) if STATE.exists() else {"steps": []}
    s["steps"] = [x for x in s["steps"] if x["step"] != step]
    s["steps"].append({"step": step,
                       "at": datetime.datetime.now().isoformat(timespec="seconds")})
    STATE.write_text(json.dumps(s, indent=2))

def save_all(step: str | None = None, name: str = "ch1_ALL", quiet: bool = False):
    """ONE zip with everything worth keeping. Call it after every real step."""
    if step:
        _mark(step)
    zpath = OUT / f"{name}.zip"
    found, missing, total = {}, [], 0
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for label, pats, _why in KEEP:
            hits = [Path(p) for pat in pats for p in glob.glob(str(REPO / pat))]
            hits = [h for h in hits if h.is_file()]
            if not hits:
                missing.append(label)
                continue
            found[label] = (len(hits), sum(h.stat().st_size for h in hits))
            total += found[label][1]
            for h in hits:
                # ★ checkpoint-*/ dirs are NEVER matched above: they hold
                #   optimizer state (hundreds of MB) and are useless for eval.
                z.write(h, str(h.relative_to(REPO)))
        if STATE.exists():
            z.write(STATE, "ch1_state.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None, "zip is corrupt"
        n = len(z.namelist())

    if not quiet:
        print(f"{'what':<12}{'files':>6}{'size':>10}   why it matters")
        print("-" * 78)
        for label, pats, why in KEEP:
            if label in found:
                c, b = found[label]
                print(f"{label:<12}{c:>6}{b/1e6:>9.1f}M   {why}")
            else:
                print(f"{label:<12}{'--':>6}{'':>10}   !! MISSING — {why}")
        print("-" * 78)
        print(f"-> {zpath}  ({zpath.stat().st_size/1e6:.1f} MB, {n} entries)")
        if missing:
            print(f"\n!! nothing found for: {', '.join(missing)}")
            print("   That is fine if you have not run that step yet — but if you")
            print("   HAVE, the file is not where this expects it.")
        print("\n   Download it from the Output panel on the right, or press")
        print("   SAVE VERSION so it survives the session ending.")
    return zpath

def restore_all(pattern: str = "ch1_ALL.zip"):
    """Attach last session's zip as a Kaggle Dataset (+ Add Input), then run this."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not hits:
        print(f"no {pattern} under /kaggle/input — attach the dataset first"); return
    for src in hits:
        with zipfile.ZipFile(src) as z:
            z.extractall(REPO)
        print(f"restored {src}")
    st = REPO / "ch1_state.json"
    if st.exists():
        shutil.copy(st, STATE)
        print("\nsteps already done last session:")
        for x in json.loads(st.read_text())["steps"]:
            print(f"   {x['at']}   {x['step']}")
    for d in sorted((REPO / "checkpoints").glob("*")):
        w = [p.name for p in d.glob("adapter_model.*")]
        print(f"   {d.name:<26}{w if w else '!! no weights'}")

def disk() -> None:
    """/kaggle/working is 20 GB. The YAGO archive alone can be several."""
    t, u, f = shutil.disk_usage(OUT)
    print(f"disk: {u/1e9:.1f} GB used, {f/1e9:.1f} GB free of {t/1e9:.1f} GB")
    big = sorted(((Path(p).stat().st_size, p) for p in glob.glob(str(OUT / "**/*"),
                  recursive=True) if Path(p).is_file()), reverse=True)[:5]
    for b, p in big:
        if b > 5e8:
            print(f"   {b/1e9:5.2f} GB  {p}   <- delete if you are done with it")

print("save_all() · restore_all() · disk()  ready")
disk()


## 1 · Tests — 2 seconds, no GPU

Each test is a worked example: it prints the actual prompt each variant produces. Read the output — it is the clearest documentation of what P0–P4 mean.

In [3]:
!python -m chapter1.test_chapter1

CHAPTER 1 TESTS — seed 66795   (reproduce: --seed 66795)
graph: 14 entities · 21 train triples

PROMPT RENDERING
  ✓ P0 bare 
      → "Is this true: Bruno, a person was born in Lyon, a city?"
  ✓ P1 type tags 
      → "Is this true: Bruno, a person [Person] was born in Lyon, a city [Location]?"
  ✓ P2 structural instruction 
      → "Is this true: Bruno, a person was born in Lyon, a city? Before answering, consider whether the two entities are compatible with the relation."
  ✓ P3 both   (types + instruction)
  ✓ P4 demonstrations 
      → "Other triples using was born in: Kofi, a person was born in Lyon, a city; Jae, a person was born in Oslo, a city; Lina, a person was born in Kyoto, a …"
  ✓ all variants render differently   all 5 variants distinct

ANONYMISATION
  ✓ removes surface forms 
      → "Is this true: entity8 was born in entity1?"
  ✓ keeps relations and structure   triples and relations preserved
  ✓ mapping is consistent   bijective and stable (7 distinct heads preserve

In [4]:
# the grid, the pre-registered interpretations, and the cost estimate
!python -m chapter1.conditions
!python -m chapter1.run --plan

CHAPTER 1 GRID

id  names  types  negatives         instances  isolates
------------------------------------------------------------------------------
A   real   no     1x random            20,000  baseline — KG-LLM's exact recipe
B   anon   no     1x random            20,000  ENTITY NAMES — everything else identical to A
C   anon   yes    1x random            20,000  TYPE INFORMATION, with names removed
D   anon   yes    1x type_consistent     20,000  negative HARDNESS (count held at 1)
E   anon   yes    6x type_consistent     70,000  negative COUNT (hardness held constant)
G   real   yes    1x random            20,000  ★★ do types help when names ARE available? The field's actual claim

------------------------------------------------------------------------------
  A: KG-LLM (ICASSP 2025): random.choice(ent_list), 1 negative per positive
  B: P12 / KG-CF: the only paper in 188 testing pretraining memorisation
  C: ★ the core new question. CATS / Knit / RealKGC all ADD types and neve

## 2 · Data — YAGO3-10 only

**YAGO3-10** carries the whole chapter: world-factual memorisation (Wikipedia people, places, clubs), strongly typed relations, 123,182 entities, and a type-tag leak already audited down to 0.513.

In [ ]:
DS = "YAGO3-10"                 # the ONLY graph this notebook runs
!python -m scripts.fetch_data --datasets {DS}

# ── other graphs, deliberately not run ───────────────────────────────────
# WN11      WordNet supersenses are the cleanest exogenous types available,
#           but memorisation there is LEXICAL, not world-factual.
# NELL-995  CATS split is INDUCTIVE and small (~476 queries) — a different
#           experiment, not a second measurement of this one.
# WN18RR    WordNet again; adds no new claim.
# FB13      tail types are fixed by the relation = the endogenous trap.
#
# !python -m scripts.fetch_data --datasets WN11 FB13 WN18RR


In [ ]:
# ═══ 2a · make the test set usable, then check it ════════════════════════
# ✋ YAGO3-10 ships 5,000 test triples with NO ±1 labels. Without negatives,
#    triple classification is IMPOSSIBLE — every gold answer becomes "No".
#    (Ranking works either way; it only needs the true tail.)
#    Writes test.original.tsv as a backup, then rewrites test.tsv.
!python -m scripts.make_test_negatives --dataset {DS} --strategy type_consistent

# file-level integrity: malformed rows, id/text mismatches, label balance
!python -m chapter1.validate --dataset {DS}

# what the graph actually looks like: degrees, relation frequencies, hubs
!python -m chapter1.profile_data --dataset {DS}


In [ ]:
# ═══ 2a-ii · MEASURE the type-tag floor — needs C/G BUILT first ══════════
# ✋ ORDERING. check_type_leak reads data/{DS}-C/built/test_instructions.json,
#    so it prints "not built yet" until the typed conditions exist. Build them
#    first, THEN measure. (Everything downstream re-uses these builds, so this
#    is not wasted work.)
for C_ in ["C", "G"]:
    !python -m chapter1.data --condition {C_} --dataset {DS}

# ★ Now the audit has something to read.
!python -m chapter1.check_type_leak --dataset {DS}

# ═════════════════════════════════════════════════════════════════════════
# ★★ STOP HERE AND DO ONE THING BY HAND.
#
#    Copy the tag-only accuracy this printed into
#        chapter1/conditions.py  ->  TYPE_TAG_FLOOR["{DS}"]
#    and push.
#
#    WHY IT MATTERS: preflight only READS that constant. It was measured on
#    the OLD test.tsv — which you just replaced. Leave it stale and every
#    typed result (C, D, E, G) silently inherits the wrong floor, and the
#    memorisation share is divided by the wrong denominator.
#
#    If the new number is >= 0.55, preflight will REFUSE the typed
#    conditions. That is correct: a one-line heuristic would explain most of
#    the result. Regenerate with --regenerate and a different strategy.
# ═════════════════════════════════════════════════════════════════════════
save_all("negatives + leak measured")


### 2b · Semantic types, straight from YAGO

YAGO was built by joining **Wikipedia** (the entities) to **WordNet** (the classes), so the types already exist — no API, no guessing. Conditions C and G go from `[_actedIn::tail]` to `[actor]` and `[film]`.

The first cell downloads the archive (several GB, run once); the second turns it into `entity2type.txt`.

In [ ]:
# ═══ 2b-i · download YAGO's own type files ═══════════════════════════════
# yago-knowledge.org does NOT publish yagoSimpleTypes.tsv on its own — the
# TSV release is one 7-Zip archive holding every "theme", and the type files
# live inside it. This downloads it, pulls out ONLY the type themes, deletes
# the archive, and then PARSE-CHECKS the result.
#
#   ⚠️ Several GB. Needs Internet ON and ~15 GB of /kaggle/working free.
#   ⚠️ Run once. It skips the download if the file is already there.
!pip -q install py7zr
!python -m scripts.download_yago_types --out /kaggle/working/yago3

# If the download fails: grab it by hand from
#   https://yago-knowledge.org/downloads/yago-3   (TSV format link)
# upload as a Kaggle Dataset, "+ Add Input", and set YAGO_TYPES below.


In [ ]:
# ═══ 2b-ii · build entity2type.txt ═══════════════════════════════════════
# ★ YAGO IS MADE FROM WORDNET. It joins Wikipedia (the entities) to WordNet
#   (the classes), so every YAGO entity already carries a class. YAGO3-10 is
#   a subset of YAGO3, so its 123,182 entities are typed BY CONSTRUCTION.
#
# The file has 5 columns, as the YAGO project documents:
#     <id_42>  <Elvis_Presley>  rdf:type  <wikicategory_American_rock_singers>
# Classes come in two layers:
#     <wordnet_actor_109765278>            the WordNet layer   ★ preferred
#     <wikicategory_1979_films>            the category layer  -> head noun
import glob
hits = sorted(glob.glob("/kaggle/working/yago3/themes/**/*ype*", recursive=True)
              + glob.glob("/kaggle/input/**/yago*ypes*.tsv", recursive=True))
assert hits, "no type file found — run the download cell, or attach a dataset"
YAGO_TYPES = hits[0]
print("using:", YAGO_TYPES)

!python -m scripts.fetch_yago_types --dataset {DS} \
    --from-yago {YAGO_TYPES} --min-coverage 0.90

# ── FALLBACK: Wikidata P31 via the enwiki title. Slower, partial coverage.
#    Use ONLY if the YAGO dump is unreachable, and lower --min-coverage
#    honestly rather than pretending the gap is not there.
# !python -m scripts.fetch_yago_types --dataset {DS} --min-coverage 0.60


In [ ]:
# ═══ what did we actually get? ═══════════════════════════════════════════
!python -m src.routing.semantic_types --dataset {DS}

# and what the prompts will look like, before spending any GPU time
from src.data.loaders import load_kg, anonymise
from src.routing.semantic_types import semantic_types
from chapter1.conditions import PROMPTS
from chapter1.data import render

kg = load_kg(DS, "data"); ty = semantic_types(kg, DS, root="data")
t  = kg.test[0]
print("\nB (anon, no types) :", render(t, anonymise(kg), PROMPTS["P0"], None, None))
print("C (anon + types)   :", render(t, anonymise(kg), PROMPTS["P1"], ty, None))
print("G (real + types)   :", render(t, kg, PROMPTS["P1"], ty, None))

# C and G MUST see the identical tag inventory or they are not a matched pair
assert semantic_types(anonymise(kg), DS, root="data") == ty, "types not anon-invariant"
print("\n✓ C and G share one tag inventory")


In [ ]:
# ═══ 2c · PREFLIGHT — no fallbacks, stops on anything wrong ══════════════
# Eight checks, each one a failure that has ALREADY happened silently in
# this project. Non-zero exit, so nothing downstream can sail past it.
#
#   1 dataset files      5 condition S really permutes (and rank.py applies it)
#   2 test labels        6 typed prompts differ from untyped
#   3 semantic types     7 C and G share ONE tag inventory
#   4 type-tag leak      8 checkpoint dir is writable
!python -m chapter1.preflight --dataset {DS} --require-semantic


In [ ]:
# ═══ 2d · DATA AUDIT (raw graph, before building anything) ═══════════════
# preflight asks "can this run start?". This asks "is the data telling the
# truth?" — a different question, and both have already failed silently here.
#
# On WN11 it reproduces every defect the paper's Threats section reports:
#   2,220 duplicate training triples · 54 duplicate test triples
#   193 self-loops · 7/10,542 negatives that are TRUE in train
#
# FAIL = stop.  WARN = a number you must REPORT, not hide.
!python -m chapter1.audit_data --dataset {DS} --skip-built


## 3 · Train — YAGO3-10 only

`--require-semantic` everywhere: if exogenous types are unavailable the build **stops** rather than quietly downgrading to induced types.

In [ ]:
# ═══ 3 · one trainer, saves after EVERY run ══════════════════════════════
import subprocess, time

def train_and_save(dataset, cond, gpu=0, prompt="P0"):
    """Train ONE condition, then immediately zip adapters + results.

    ★ `prompt` is passed through because it is part of a run's IDENTITY.
      chapter1/run.py used to ignore it: `--prompt P6` read the P0 instances
      and then OVERWROTE the P0 adapter with the result. Non-P0 prompts now
      live at checkpoints/ch1-{ds}-{cond}-{prompt}.

    Saving per-run rather than per-session is the whole point: a session that
    dies after run 3 of 5 still leaves 3 usable adapters on disk. Training is
    the only expensive step here — everything downstream is cheap and
    re-runnable, but only if the adapters survive.
    """
    t0 = time.time()
    rc = subprocess.run(
        f"CUDA_VISIBLE_DEVICES={gpu} python -m chapter1.run "
        f"--dataset {dataset} --train --condition {cond} "
        f"--prompt {prompt}", shell=True).returncode
    mins = (time.time() - t0) / 60
    label = f"{dataset} {cond}" + ("" if prompt == "P0" else f" [{prompt}]")
    if rc:
        print(f"\n✗ {label} FAILED (rc={rc}) after {mins:.0f} min — not saving")
        return False
    print(f"\n✓ {label} trained in {mins:.0f} min")
    save_all()
    return True

def pair(a, b):
    """Two INDEPENDENT jobs, one per T4. Never DataParallel."""
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()


In [ ]:
# --require-semantic: refuse the INDUCED fallback. Any run whose numbers go
# in the paper must use this — it turns a silent downgrade into a hard stop.
for C_ in ["A", "S", "B", "C", "G"]:
    !python -m chapter1.data --condition {C_} --dataset {DS} --require-semantic

for C_ in ["A", "S", "B", "C", "G"]:
    if not train_and_save(DS, C_):
        print(f"stopping at {C_} — fix it before continuing"); break


In [ ]:
# ═══ 3b · DATA AUDIT (after building — the arms, types and pools) ════════
# Now that the instances exist, check the things that only exist once built:
#   · do the conditions actually produce DIFFERENT prompts?
#   · do their test sets align row-for-row (or the gap compares different triples)
#   · are the types invariant under anonymise AND permute (C/G matched pair)
#   · does every candidate pool contain the gold, and exactly 50 entries
!python -m chapter1.audit_data --dataset {DS} --conditions A S B C G


### Other graphs *(not run)*

In [ ]:
# ═══ other graphs — NOT RUN ══════════════════════════════════════════════
# Kept so the pipeline is visibly dataset-agnostic. Each needs its own
# preflight to pass before any training.
#
# ── WN11 (WordNet supersenses; lexical memorisation) ─────────────────────
# ✋ GATE: WN11's tag-only rule scores 0.568 > 0.55, so preflight FAILS
#    until the test negatives are regenerated.
# !python -m scripts.make_test_negatives --dataset WN11 --strategy type_consistent --regenerate
# !python -m chapter1.preflight --dataset WN11 --require-semantic
# for C_ in ["A","S","B","C","G"]:
#     !python -m chapter1.data --condition {C_} --dataset WN11 --require-semantic
#     train_and_save("WN11", C_)
#
# ── NELL-995 (ontology types free in the id; INDUCTIVE split) ────────────
# !python -m scripts.convert_cats --src /kaggle/input/datasets/alwayshigh10/cats-data/NELL-995-subset-inductive --out data/NELL-995-ind
# !python -m scripts.make_test_negatives --dataset NELL-995-ind --strategy type_consistent
# !python -m chapter1.preflight --dataset NELL-995-ind --require-semantic


## 4 · Evaluate — both test sets, always

Each model is scored on the **real** and the **anonymised** test set. The **gap** is the result; a single accuracy number cannot express the claim.

This also emits the seen/unseen split and the calibration-by-familiarity block for free.

In [ ]:
# score every trained condition on BOTH test sets. The gap is the result.
# S is scored on its OWN permuted test set (evaluate.py bug fix) -- not on A's.
for C_ in ["A", "S", "B", "C", "G"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.run --dataset {DS} --evaluate --condition {C_}

save_all('evaluation (both test sets)')      # results/*.json are packed too


In [ ]:
!python -m chapter1.analysis --dataset {DS}
# ...and the BALANCED familiarity split. analysis.py reports RAW accuracy,
# which the differing bucket base rates make misleading -- use this one.
!python -m chapter1.seen_unseen --dataset {DS}


In [ ]:
# ═══ 4c · the full report — per-relation, confusion, risk–coverage ═══════
# chapter1.analysis gives the gap. This gives everything the paper's
# Threats and per-relation discussion depend on:
#     precision / recall / F1 per class + confusion matrix
#     degenerate check   is it just answering the same thing every time?
#     per-relation       WHICH relations carry the score (EIR 28.96 on WN11 —
#                        one relation dominates, so the aggregate misleads)
#     risk–coverage      the abstention curve Chapter 4 consumes
#     McNemar            significance against a named baseline
for C_ in ["A", "S", "B", "C", "G"]:
    !python -m chapter1.report --dataset {DS} --condition {C_}

save_all("full report")


### 4b · Beside the published numbers

KG-LLM's Table II, with our row added — and a hard rule about which cells may be compared. Also adds **AUC**, **macro-F1** and **McNemar**, none of which the evaluation reported before.

In [ ]:
# ═══ 4b · OUR ROW IN KG-LLM's TABLE II  +  the metrics we were missing ═══
# Their Table II lists 21 methods on WN11/FB13. Adding our row is the clearest
# proof the reproduction is faithful — but only where the comparison is valid.
#
#   ✓ COMPARABLE   WN11 triple classification. Same dataset, same SHIPPED ±1
#                  labels, same task, same metric.
#   ✗ NOT          YAGO3-10 classification (our negatives are GENERATED),
#                  FB13 (never ran it), and ANY link-prediction number
#                  (theirs is full-ranking over 123,182 entities, ours 50-way).
#
# --extra adds three things the evaluation never reported:
#   AUC       separates positives from negatives at ANY threshold. This is what
#             settles condition E: it scored 0.5010 at argmax, but if AUC is
#             high it DID learn and only the 1:6 training prior miscalibrated it.
#   macro-F1  standard in this literature; accuracy alone hides class collapse.
#   McNemar   the RIGHT test for two models on the SAME rows. Exact, so it
#             needs no repeated seeds — which we do not have.
!python -m chapter1.compare --dataset WN11 --extra

# ✋ WN11 is the only comparable cell. For YAGO3-10 it prints our rows alone
#    and refuses to place them in their table:
!python -m chapter1.compare --dataset {DS} --extra

save_all("comparison vs KG-LLM Table II")


## 5 · Link prediction — is this really KGC?

Triple classification completes nothing. Here the classifier becomes a **ranker**: score every candidate tail by `P(Yes | h, r, t)` and sort. No retraining — the model was trained to emit exactly that judgement.

★ **This makes MRR computable**, which the spec had recorded as impossible under generative decoding.

⚠️ **50-way, filtered.** Not comparable to full-ranking numbers (R12). Say so in every caption.

In [ ]:
# A / S / B = the three arms of the decomposition. C only if you want the
# typed ranking row too.
for C_ in ["A", "S", "B"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
        --dataset {DS} --condition {C_} --adapter checkpoints/ch1-{DS}-{C_}

save_all('ranking A/S/B')


### 5b · The untuned baseline — and a bug fix you need to read

`rank.py` never applied `cond.shuffle`, so **condition S was ranked on the real graph**. The S adapter was trained on a deranged world and then scored on undamaged names — a train/test mismatch, not the permuted-name control the paper describes. `m(S)=0.2974`, and therefore the 71.5 % binding term, came from that. Fixed; S must be re-ranked.

In [ ]:
# ═══ 5b · untuned ranking baseline  +  the S re-run ══════════════════════
DS = "YAGO3-10"        # the graph that carries the decomposition

# ── 1. UNTUNED. Omit --adapter. Inference only, nothing to train. ────────
#
#   This is the highest-value run left in the chapter. If the UNTUNED model
#   already ranks far above chance on real names and collapses when they are
#   removed, then the name->node binding lives in the PRETRAINED backbone,
#   not in your LoRA. That reframes the whole result: "a 1.5B model has no
#   capacity to do anything but memorise" stops being an objection, because
#   the claim is no longer about capacity or about your recipe at all.
#
#   It also corroborates the familiarity split (+0.0036) from a completely
#   independent direction.
for C in ["A", "S", "B"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
        --dataset {DS} --condition {C} --tag ch1rank-{DS}-{C}-untuned

# ── 2. S, re-ranked with the permutation ACTUALLY APPLIED ───────────────
#
#   rank.py never called shuffle_surface_forms, so --condition S ranked on
#   the REAL graph: the S adapter, trained on a deranged world, was scored
#   on undamaged names. That is a train/test mismatch, not the permuted-name
#   control, and m(S)=0.2974 -- hence the 71.5% "binding" term -- came from
#   it. Fixed in chapter1/rank.py.
#
#   ** Check the printed line says  surface form: PERMUTED  **
#   ** and that the example query head is NOT a real name.  **
!CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
    --dataset {DS} --condition S --adapter checkpoints/ch1-{DS}-S \
    --tag ch1rank-{DS}-S-fixed

save_all('untuned baseline + S re-rank')        # cheap insurance


In [ ]:
# ═══ 5c · bootstrap CIs on the decomposition (CPU, seconds) ══════════════
# Consumes the full 500-query dumps that rank.py now writes (it used to
# truncate to 200, which widened every interval by ~1.6x for nothing).
import json, random
from pathlib import Path

CHANCE, B = 0.0900, 10000
random.seed(42)

def load(tag):
    p = Path("results") / f"{tag}.json"
    if not p.exists(): return None
    return {(r["head"], r["relation"], r["tail"]): 1.0 / r["rank"]
            for r in json.load(p.open())["ranks"]}

arms = {k: load(t) for k, t in
        [("A", f"ch1rank-{DS}-A-P0"), ("S", f"ch1rank-{DS}-S-fixed"),
         ("B", f"ch1rank-{DS}-B-P0")]}
missing = [k for k, v in arms.items() if v is None]
if missing:
    raise SystemExit(f"missing arms {missing} - run 5b first")

keys = sorted(set.intersection(*(set(v) for v in arms.values())))
n = len(keys)
rr = {k: [v[q] for q in keys] for k, v in arms.items()}
idx = [[random.randrange(n) for _ in range(n)] for _ in range(B)]
mean = lambda xs, ix: sum(xs[i] for i in ix) / len(ix)

print(f"{n} aligned queries, {B:,} paired resamples\n")
print(f"{'arm':<4}{'MRR':>9}{'95% CI':>22}")
for k in "ASB":
    v = sorted(mean(rr[k], ix) for ix in idx)
    print(f"{k:<4}{sum(rr[k])/n:>9.4f}   [{v[int(.025*B)]:.4f}, {v[int(.975*B)]:.4f}]")

print(f"\n{'component':<22}{'share':>8}{'95% CI':>20}")
tot = [rr["A"][i] - CHANCE for i in range(n)]
for lab, num in [("binding  A->S",    [rr["A"][i]-rr["S"][i] for i in range(n)]),
                 ("readability S->B", [rr["S"][i]-rr["B"][i] for i in range(n)]),
                 ("residual B-chance",[rr["B"][i]-CHANCE      for i in range(n)]),
                 ("MEMORISATION A->B",[rr["A"][i]-rr["B"][i] for i in range(n)])]:
    v = sorted(sum(num[i] for i in ix)/sum(tot[i] for i in ix) for ix in idx)
    print(f"{lab:<22}{100*sum(num)/sum(tot):>7.1f}%"
          f"   [{100*v[int(.025*B)]:.1f}%, {100*v[int(.975*B)]:.1f}%]")


### 5d–5e · Qualitative samples, and a plain-English report

`--story` turns the whole result into sentences:

```
> Where was Luciano Quadros da Silva born?
  The true answer is Porto Alegre.

  with the real names            -> Porto Alegre        RIGHT, first answer
  with the names swapped around  -> Kirklareli Province  right answer was #3
  with the names hidden          -> University of Szeged right answer was #49
```

and translates the metrics: **71 out of 100 on the first try** with names, **5 out of 100** without. Random guessing gets 2.

In [ ]:
# ═══ 5d · what the model ACTUALLY answers ════════════════════════════════
# No GPU — reads results/*.json.
!python -m chapter1.showcase --dataset {DS} --n 8 --seed 42 --task both --top5

# ═══ 5e · the SAME result, written in plain English ══════════════════════
# For sharing. No MRR, no Hits@K — real questions, real answers, and the
# metrics translated ("71 out of 100 on the first try" instead of 0.714).
# Writes results/ch1_story_{DS}.md — paste straight into an email or slides.
!python -m chapter1.showcase --dataset {DS} --n 5 --seed 42 --story

save_all('showcase + plain-English story')


## 6 · Context variants — P5, P6, P7

Inference only, no retraining. **P6 is KG-LLM's own K=5 neighbours**, which we had never reproduced — condition A is their recipe *without* it. Our guard is stricter than theirs: they exclude the target entity, we also drop every edge on the query relation, which otherwise names a true answer outright.

In [ ]:
# ═══ 6a · CONTEXT VARIANTS P5 / P6 / P7 — inference only, NO retraining ═══
# These re-score the checkpoints you ALREADY trained. ~20 min each.
#
#   P5  relation description   what the relation MEANS (hand-written)
#   P6  K=5 neighbours         ★ KG-LLM's OWN mechanism — we never had it.
#                                Their biggest single gain: YAGO3-10 Hits@1
#                                0.0949 -> 0.1330 (their full-ranking numbers).
#   P7  paths                  the chain linking head to candidate
#
# ✋ co-occurrence confidence is DELIBERATELY ABSENT. Counting how often a pair
#    appears uses the very edges we are predicting — the same endogenous trap
#    that made induced types score 62.4% with no model at all.
#
# ⚠️ ONLY INCREASES ARE INTERPRETABLE on a tuned checkpoint: it only ever saw
#    P0, so a DROP may be distribution shift rather than inability.
#
# THE QUESTION (pre-registered in chapter1/conditions.py):
#    every published context method is measured with names PRESENT, so context
#    and familiarity are confounded in all of them. A is names-on, B is
#    names-off — running the same prompt on both separates them.
for P in ["P5", "P6", "P7"]:
    for C_ in ["A", "B"]:
        !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
            --dataset {DS} --condition {C_} --prompt {P} \
            --adapter checkpoints/ch1-{DS}-{C_} --tag ch1rank-{DS}-{C_}-{P}
        save_all(f"context {P} on {C_}", quiet=True)   # after EVERY run

# WATCH TWO LINES in every run:
#   [context] leak check: 0 leaks in 300 queries; guard removed the gold ...
#   [context] P7: 143/500 queries got a non-empty block (28.6%)   <- COVERAGE
# A variant whose block is empty IS P0. With low coverage, "no effect" is a
# statement about availability, not about whether context helps.


### 6b · Training on the prompt *(optional, after 6a)*

Inference-only can't tell *"the idea is bad"* from *"the format is unfamiliar"*. Training on it can. **P6 only** — it's the one with a published number to check against.

In [ ]:
# ═══ 6b · TRAIN on the context prompt — only after 6a shows a signal ═════
# 6a re-scored existing checkpoints, so a DROP there could just be an
# unfamiliar format. Training on the prompt removes that excuse: a drop then
# means the idea does not help.
#
# ★ P6 ONLY, and here is why. P5 and P7 have no published number to check
#   against — a trained result there is one more ablation. P6 has KG-LLM's
#   Table IV sitting there: YAGO3-10 Hits@1 0.0949 -> 0.1330 WITH neighbours.
#   Training it makes condition A their ACTUAL best system, not just their base.
#
# ~2 training runs, ~1.5 GPU-h. Do NOT start this before 6a.
for C_ in ["A", "B"]:
    !python -m chapter1.data --condition {C_} --dataset {DS} --prompt P6 \
        --require-semantic --min-context 0.5
    if not train_and_save(DS, C_, prompt="P6"):
        print(f"stopping at {C_}"); break

# score them on BOTH test sets (classification), then rank them.
# ★ Without this the P6 story exists only in ranking and section 4b has no
#   P6 row to put beside KG-LLM's Table II.
for C_ in ["A", "B"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.run \
        --dataset {DS} --evaluate --condition {C_} --prompt P6

for C_ in ["A", "B"]:
    !CUDA_VISIBLE_DEVICES=0 python -m chapter1.rank \
        --dataset {DS} --condition {C_} --prompt P6 \
        --adapter checkpoints/ch1-{DS}-{C_}-P6 --tag ch1rank-{DS}-{C_}-P6trained
    save_all(f"trained P6 on {C_}")

# ✋ THE BUILD REFUSES if the context block is empty for >50% of instances.
#    That guard exists because `--prompt P6` USED to build P0 prompts under a
#    P6 label — render() takes a context argument and the builder never passed
#    it. A 40-minute run would have produced an adapter that never saw a
#    single neighbour, and nothing would have said so.
#
#    On WN11 the guard measured P6 at 93.0% coverage and P7 at 5.3% — P7 is
#    REFUSED by default, because "paths do not help" and "paths do not exist"
#    are not the same finding. Override only with --min-context, and then
#    report the number.


In [ ]:
# ═══ 6c · read the result — the four outcomes, written in advance ════════
# Compare, for the SAME arm, P0 against P6:
#     ch1rank-{DS}-A-P0.json        vs   ch1rank-{DS}-A-P6trained.json
#     ch1rank-{DS}-B-P0.json        vs   ch1rank-{DS}-B-P6trained.json
# ★ --tags: showcase reads the TUNED P0 arms by default. To see the
#   trained-on-P6 runs you must ask for them by tag, and every arm must
#   supply the SAME tag or you are comparing different systems.
!python -m chapter1.showcase --dataset {DS} --n 6 --seed 42 --story
!python -m chapter1.showcase --dataset {DS} --n 6 --seed 42 --story --tags P6trained

# ┌──────────────────────────┬────────────────────────────────────────────┐
# │ P6 helps A, not B        │ ★★ the neighbours worked because they were │
# │                          │ FAMOUS NAMES, not because of structure.    │
# │                          │ KG-LLM's +40% is another surface-form gain.│
# ├──────────────────────────┼────────────────────────────────────────────┤
# │ P6 helps B, not A        │ ★★ context is REDUNDANT when the name      │
# │                          │ answers the question and LOAD-BEARING when │
# │                          │ it does not. Every published context gain   │
# │                          │ was measured in the wrong regime.          │
# ├──────────────────────────┼────────────────────────────────────────────┤
# │ P6 helps neither         │ we failed to reproduce KG-LLM's gain at    │
# │                          │ 1.5B — say so, and give the coverage.      │
# ├──────────────────────────┼────────────────────────────────────────────┤
# │ P6 helps both equally    │ genuine structural signal, independent of  │
# │                          │ names. The strongest positive result here. │
# └──────────────────────────┴────────────────────────────────────────────┘
#
# ⚠️ Read the CI, not the point estimate — cell 5c has the paired bootstrap.
#    And quote the context coverage from the manifest beside every number.
save_all("P6 comparison complete")


## 7 · Package — save before the session dies

In [ ]:
# ═══ 7 · PACKAGE — run this ANY time you think the session might die ═════
# save_all() already keeps exactly the four things worth keeping, ordered by
# how painful they are to lose:
#     adapters   GPU-hours, cannot be recreated without retraining
#     types      a multi-GB download plus a full pass over the YAGO dump
#     results    every number, every figure input, the story report
#     manifests  provenance: seed, condition, type source, positive rate
# Trainer's checkpoint-*/ folders are NEVER included — hundreds of MB of
# optimizer state that evaluation never touches.
save_all("final package")
disk()

print("\nfiles waiting in the Output panel on the right:")
!ls -lh /kaggle/working/*.zip
print("\n★ Kaggle DELETES /kaggle/working unless you press 'Save Version'.")
print("  Download ch1_ALL.zip now, or Save Version. Next session, attach it")
print("  as a Dataset and run restore_all() — you skip straight to evaluation.")
